In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

# 1.加载环境变量
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv()

# 2.初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

# 3.定义工具，用MCP获取工具

# 3.1.定义mcp client
client = MultiServerMCPClient(
    {
        "time-mcp": {
            "transport": "stdio",
            "args": [
                "-y",
                "time-mcp"
            ],
            "command": "npx"
        }
    }
)
# 3.2.用client拉取tool
tools = await client.get_tools()


# 4.创建Agent，绑定模型和工具
agent = create_agent(
    model=model,
    tools=tools
)

# 5.由于MCP的Tool是异步的，所以必须用ainvoke调用Agent，是异步调用
response = await agent.ainvoke(
    {"messages": [HumanMessage("现在是什么时间")]}
)
print(response)

{'messages': [HumanMessage(content='现在是什么时间', additional_kwargs={}, response_metadata={}, id='492ad98e-6b6d-46b8-9e53-0bc82507ac9c'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 946, 'total_tokens': 973, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 896, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 896, 'prompt_cache_miss_tokens': 50}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '71f5bedd-ef95-4273-b1ee-e06535ad8e71', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a047a7-24d3-74f3-b54b-1f268d309a9e-0', tool_calls=[{'name': 'current_time', 'args': {}, 'id': 'call_00_NkhbOHEBVPAGssPeGLmx0248', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 946, 'output_tokens':